# Solutions 08: The Teacher Server, Measured

This notebook solves the four exercises of Lab 08
(`labs/lab-08-teacher-server-and-measurement.ipynb`). Execution status: exercise 4 is fully
live arithmetic; exercises 1, 2, and 3 run their reasoning live (roofline ladders, the
co-tenancy model, and a latency-instrumented mock server measured for real) and gate the
parts that need an actual vLLM server behind `RUN_SERVER = False`, the lab's own pattern.

Attempt the exercises first. Exercises 1, 2, and 4 are answerable with a pencil before any
server exists, and doing them that way is the skill the lab is teaching.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, time, threading
sys.path.insert(0, "../code")

import numpy as np
import torch

from kd_pipeline import (set_seed_everywhere, MemoryPlan, infer_gb, full_ft_gb,
                         kv_cache_gb, bandwidth_bound_decode_tps)

RUN_SERVER = False        # <-- flip on the training box (needs a live vLLM server)
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
BW = 273.0                # GB/s, the course's bandwidth figure
print(f"torch {torch.__version__} | device: {device} | RUN_SERVER: {RUN_SERVER}")

torch 2.13.0+cpu | device: cpu | RUN_SERVER: False


## Exercise 1: The quantization ladder

**The exercise, restated.** Serve the same teacher in bf16 (2 bytes per parameter), 8-bit
(1 byte), and 4-bit (half a byte); measure the decode curve for each. Does throughput scale
with bytes-per-parameter the way the roofline predicts, or does kernel quality dominate?

**The approach.** The measurement needs a server, so it is gated. The prediction the
measurement would be checked against is pure roofline arithmetic, and building it first is
not optional busywork: without the predicted ladder you cannot tell "kernel quality
dominates" apart from "I mis-measured". The roofline says decode tokens per second is capped
at bandwidth divided by bytes read per token, and for a dense model the bytes read per token
are the whole weight footprint, so halving bytes-per-parameter must exactly double the
ceiling. The live cell builds that ladder for four teacher sizes, asserts the exact 4x ratio
between bf16 and 4-bit, and runs the memory side of the same ladder: which rungs fit this
box at all, and which fit *alongside* a student training run, using the lab's own KV-cache
arithmetic (the stored attention keys and values, which grow with sequence length and
concurrency) for a 32B-class serving configuration.

In [2]:
# Live: the predicted ladder and the fit table.
SIZES = [8.0, 20.0, 32.0, 70.0]
RUNGS = [("bf16", 2.0), ("8-bit", 1.0), ("4-bit", 0.5)]

print(f"{'model':>7} | " + " | ".join(f"{n:>16}" for n, _ in RUNGS))
tps = {}
for pb in SIZES:
    row = []
    for name, bpp in RUNGS:
        tps[(pb, name)] = bandwidth_bound_decode_tps(pb, BW, bytes_per_param=bpp)
        row.append(f"{infer_gb(pb, bpp):5.0f} GB {tps[(pb, name)]:6.1f}/s")
    print(f"{pb:>6}B | " + " | ".join(f"{c:>16}" for c in row))

for pb in SIZES:
    assert abs(tps[(pb, '4-bit')] / tps[(pb, 'bf16')] - 4.0) < 1e-9, \
        "the roofline prediction is an exact 4x from bf16 to 4-bit"
    assert abs(tps[(pb, '8-bit')] / tps[(pb, 'bf16')] - 2.0) < 1e-9

# The memory side: a 32B teacher at each rung, co-tenant with the 360M student's
# full fine-tune, on the 128 GB box with the course's 15% headroom rule.
kv = kv_cache_gb(n_layers=64, n_kv_heads=8, head_dim=128, seq_len=4096, batch=16)
print(f"\nKV cache, 32B-class geometry, 16 seqs x 4096: {kv:.1f} GB (same at every rung)")
for name, bpp in RUNGS:
    mp = (MemoryPlan(total_gb=128.0)
          .add(f"teacher 32B {name}", infer_gb(32.0, bpp))
          .add("KV cache 16x4096", kv)
          .add("student 360M full FT + activ.", full_ft_gb(0.36) + 4.0))
    print(f"  32B {name:>5}: planned {mp.planned_gb:6.1f} GB -> fits: {mp.fits}")
    assert mp.fits, "every 32B rung must co-tenant on this box; the ladder is measurable"
assert infer_gb(70.0, 2.0) > 128.0, "70B bf16 does not fit even alone: no bf16 rung exists"
print(f"  70B bf16 : {infer_gb(70.0, 2.0):.0f} GB weights alone > 128 GB, unmeasurable here")
print("\nprediction on record: 2x per halving of bytes, or kernels are eating the difference")

  model |             bf16 |            8-bit |            4-bit
   8.0B |    16 GB   17.1/s |     8 GB   34.1/s |     4 GB   68.2/s
  20.0B |    40 GB    6.8/s |    20 GB   13.7/s |    10 GB   27.3/s
  32.0B |    64 GB    4.3/s |    32 GB    8.5/s |    16 GB   17.1/s
  70.0B |   140 GB    1.9/s |    70 GB    3.9/s |    35 GB    7.8/s

KV cache, 32B-class geometry, 16 seqs x 4096: 17.2 GB (same at every rung)
  32B  bf16: planned   90.9 GB -> fits: True
  32B 8-bit: planned   58.9 GB -> fits: True
  32B 4-bit: planned   42.9 GB -> fits: True
  70B bf16 : 140 GB weights alone > 128 GB, unmeasurable here

prediction on record: 2x per halving of bytes, or kernels are eating the difference


In [3]:
# Gated: the measurement, the lab's B-2 decode harness once per served rung.
if RUN_SERVER:
    import requests
    def measure_rung(server, model_name, batches=(1, 8), new_tokens=256):
        out = {}
        for b in batches:
            prompts = [[1000 + i] * 32 for i in range(b)]
            def one():
                for p in prompts:
                    requests.post(f"{server}/v1/completions", json={
                        "model": model_name, "prompt": p,
                        "max_tokens": new_tokens}, timeout=600)
                return b * new_tokens
            times = []
            for _ in range(4):
                t0 = time.perf_counter(); toks = one()
                times.append(time.perf_counter() - t0)
            out[f"{b}x{new_tokens}"] = round(toks / sorted(times[1:])[1], 1)
        return out

    # Serve each rung in turn from a terminal, then run this cell per rung:
    #   vllm serve <teacher>            --port 8000   (bf16)
    #   vllm serve <teacher-8bit>       --port 8000
    #   vllm serve <teacher-AWQ-4bit>   --port 8000
    ladder = {}
    ladder["4-bit"] = measure_rung("http://127.0.0.1:8000", "served-teacher")
    with open("../runs/lab08/quant_ladder.json", "w") as f:
        json.dump(ladder, f, indent=2)
    print(json.dumps(ladder, indent=2))
else:
    print("RUN_SERVER=False: the per-rung measurement compiled but did not execute.")

RUN_SERVER=False: the per-rung measurement compiled but did not execute.


**Interpretation.** The live ladder is the null hypothesis the gated measurement would
test: exact 2x aggregate decode ceiling per halving of bytes-per-parameter, asserted to
machine precision because it is the same division with a different denominator. The fit
table adds the constraint that decides what is even measurable here: all three 32B rungs
co-tenant with the student's training footprint on the 128 GB box (bf16 is the tight one at
90.9 GB planned against a 108.8 GB post-headroom budget), while a 70B bf16 rung does not
exist on this hardware at all, 140 GB of weights against 128 GB of memory.

The measurement did not run (`RUN_SERVER=False`, one line of honesty). Expected result,
grounded in the lab's Part A·1 and Part C: measured rungs land *under* their ceilings
(attention math, KV reads, and scheduling take time the bound ignores), and the honest
comparison is of *ratios*, not absolutes. If bf16-to-4-bit measures close to 4x, bandwidth
rules and quantization is nearly free throughput. The likelier finding on a young platform,
and the reason the exercise says "kernel quality": the 4-bit rung measuring well short of
its predicted multiple, for example 2 to 3x over bf16 instead of 4x, because dequantization
kernels for the format are immature, so each weight read costs extra compute the roofline
does not model. The lab's Part C states the signature explicitly: a prefill-to-decode ratio
over 100:1 means decode is starved and the quantization kernel is the suspect. Refuting
evidence for the roofline itself would be any rung measuring *above* its ceiling, which, per
A·1, means the footprint number is wrong (a MoE teacher, exercise 4's subject), not that the
bound broke.

## Exercise 2: The co-tenancy frontier

**The exercise, restated.** Sweep the server's memory fraction (`--gpu-memory-utilization`
in {0.4, 0.55, 0.7}) while a fixed student training job runs beside it; plot both processes'
throughput and find the allocation that maximizes *joint* progress. That number is the box's
real teacher budget.

**The approach.** The real sweep needs both processes live, so it is gated. But the shape
of the answer, and where the best point sits, comes from a small model built live from
quantities the course already has. Give the teacher a fraction f of the 128 GB pool. Two
opposing lines follow:

- *Teacher scoring throughput rises with f.* The teacher's weights are a fixed cost; every
  GB above them becomes KV cache, and KV capacity is concurrency (A·1's arithmetic: about
  1.07 GB per 4096-token sequence at 32B-class geometry). More concurrent sequences means
  more scoring tokens per second, linearly, until compute saturates.
- *Student throughput falls with f.* On unified memory both processes draw from one
  bandwidth pool, and the lab's Part C names the contention explicitly. The simplest
  bandwidth-sharing assumption: the student's achievable tokens per second scales with the
  share of the machine not promised to the teacher, S(f) = S0 times (1 - f).

Joint progress is the smaller of what the student can train and what the teacher can score
for it (here, r = 4 teacher-scored tokens per student token, a scoring-heavy on-policy
setup). The minimum of a rising line and a falling line is maximized where they cross, and
that crossing has a closed form. The cell computes it, checks it against a brute-force grid,
and evaluates the exercise's three sweep points. Every constant is a stated modeling
assumption, not a measurement; the analytic structure is what transfers to the real sweep.

In [4]:
# Live: the joint-progress model and its analytic knee.
M = 128.0                 # GB, the whole box
W_T = 18.0                # GB, 32B 4-bit teacher weights + server overhead
KV_SEQ = kv_cache_gb(64, 8, 128, 4096, batch=1)     # GB per concurrent sequence
C_SEQ = 60.0              # teacher scoring tok/s per concurrent sequence (assumption)
S0 = 1200.0               # student training tok/s with the box to itself (assumption)
R = 4.0                   # teacher tokens scored per student token (scoring-heavy)

def student_tps(f):                       # bandwidth-sharing assumption
    return S0 * (1.0 - f)
def teacher_score_tps(f):                 # KV-limited concurrency
    return max(0.0, C_SEQ * (f * M - W_T) / KV_SEQ)
def joint(f):                             # progress = the binding side
    return min(student_tps(f), teacher_score_tps(f) / R)

# Analytic knee: S0(1-f) = C_SEQ (fM - W_T) / (KV_SEQ * R), solved for f.
a = C_SEQ / (KV_SEQ * R)
f_star = (S0 + a * W_T) / (S0 + a * M)
print(f"KV per sequence: {KV_SEQ:.2f} GB | analytic knee f* = {f_star:.3f} "
      f"-> joint {joint(f_star):.0f} student tok/s")

grid = [W_T / M + 0.005 + 0.005 * i for i in range(int((0.95 - W_T / M) / 0.005))]
f_grid = max(grid, key=joint)
print(f"grid argmax        f  = {f_grid:.3f} -> joint {joint(f_grid):.0f} student tok/s")
assert abs(f_grid - f_star) <= 0.0075, "the brute-force optimum must sit at the crossing"

print(f"\n{'f':>5} {'student tok/s':>14} {'teacher score/s':>16} {'joint':>7}  binding side")
for f in (0.40, 0.55, 0.70):
    s, t = student_tps(f), teacher_score_tps(f)
    side = "teacher-starved" if t / R < s else "student-starved"
    print(f"{f:>5.2f} {s:>14.0f} {t:>16.0f} {joint(f):>7.0f}  {side}")

assert joint(0.55) > joint(0.40) and joint(0.55) > joint(0.70), \
    "of the swept points, 0.55 must win: it sits nearest the crossing"
assert joint(f_star) >= joint(0.55), "the knee itself is at least as good as any swept point"
print(f"\nknee at f* = {f_star:.2f}: below it the teacher starves the student of scores,")
print("above it the teacher's memory grab starves the student of bandwidth")

KV per sequence: 1.07 GB | analytic knee f* = 0.486 -> joint 617 student tok/s
grid argmax        f  = 0.486 -> joint 617 student tok/s

    f  student tok/s  teacher score/s   joint  binding side
 0.40            720             1855     464  teacher-starved
 0.55            540             2928     540  student-starved
 0.70            360             4001     360  student-starved

knee at f* = 0.49: below it the teacher starves the student of scores,
above it the teacher's memory grab starves the student of bandwidth


In [5]:
# Gated: the real sweep, one server restart per point with the student job fixed.
if RUN_SERVER:
    import requests
    results = {}
    for f in (0.4, 0.55, 0.7):
        # restart the server from a terminal at this fraction:
        #   vllm serve Qwen/Qwen3-32B-AWQ --max-model-len 4096 \
        #       --gpu-memory-utilization {f} --max-num-seqs 16 --port 8000
        input(f"server up at --gpu-memory-utilization {f}? press enter")
        assert requests.get("http://127.0.0.1:8000/v1/models", timeout=10).ok
        # (1) teacher side: the lab's measure_prefill at batch 16 x 512
        # (2) student side: time 50 optimizer steps of the Lab 04 loop running
        #     concurrently, report steps/sec
        # results[f] = {"teacher_prefill_tps": ..., "student_steps_per_s": ...}
    with open("../runs/lab08/cotenancy_frontier.json", "w") as fjson:
        json.dump(results, fjson, indent=2)
else:
    print("RUN_SERVER=False: the live sweep compiled but did not execute.")

RUN_SERVER=False: the live sweep compiled but did not execute.


**Interpretation.** The model puts the knee at the printed f* of roughly 0.49 under the
stated assumptions, and the three-point sweep reads exactly as the exercise's phrase "the
frontier" suggests: at 0.40 the system is teacher-starved (the teacher's leftover KV after
its 18 GB of weights supports too little concurrency, so the student idles waiting for
scores), at 0.70 it is student-starved (the teacher scores far more than needed while the
student's bandwidth share drags), and 0.55 wins among the swept points because it sits
nearest the crossing, which the grid search confirms against the closed form. The structural
lesson survives any change of constants: joint progress is a min of a rising and a falling
function of f, so the optimum is at their crossing, not at either process's private optimum,
and the crossing moves right when scoring demand r grows and left when the teacher's
per-sequence scoring rate improves.

The real sweep is gated (`RUN_SERVER=False`; the table above is a model, not a
measurement). What the live version should show, grounded in Part C: the same unimodal shape
with the peak in the 0.45 to 0.60 band for a scoring-heavy run, and the Part C failure
signature on the far side, training-start throughput collapse, growing worse with f as the
processes fight over one memory pool. What would refute the model's *shape*: joint progress
monotone in f all the way to 0.7, which would mean scoring demand r is much higher than
assumed (the student is always the starved side); monotone *down* from 0.4 means the
opposite. Either way the fix is re-fitting two constants, not rethinking the frontier, and
the fitted crossing becomes the box's real teacher budget, the number this exercise is
named after.

## Exercise 3: Score-batch amortization

**The exercise, restated.** Measure student steps per second at a scoring batch of {1, 4,
16} rollouts per request. Where does the curve knee? Reconcile with the prefill curve from
the lab's B·2.

**The approach.** The real measurement needs the vLLM server and is gated. What can be
measured live, for real, is the client-side half of the phenomenon, which is where most of
the effect lives: every scoring request pays a fixed per-request cost (connection handling,
HTTP parsing, scheduling, queueing) before any token gets scored, so sending one rollout per
request pays that toll sixteen times per optimizer step while sending sixteen rollouts pays
it once. The lab's A·2 mock server pattern extends naturally: the mock below charges an
explicit 50 ms per request plus 0.1 ms per token, constants chosen to caricature a busy
co-tenant server, and a simulated training step that needs 16 rollouts of 64 tokens scored
is timed against it at the three batch sizes. Because the mock's cost model is known
exactly, the measured curve can be checked against the analytic prediction, which is the
discipline the lab's benchmark harness section teaches: never trust a harness you have not
run against a known workload.

In [6]:
# Live: a latency-instrumented mock server, measured for real.
import http.server, socketserver, requests

REQ_LATENCY_S = 0.050        # fixed cost per request (the amortizable part)
PER_TOKEN_S = 0.0001         # marginal cost per scored token (does not amortize)
ROLLOUTS_PER_STEP, ROLLOUT_LEN = 16, 64

class MockScoringServer(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        n_tokens = sum(len(p) for p in body["prompts"])
        time.sleep(REQ_LATENCY_S + PER_TOKEN_S * n_tokens)   # the priced work
        out = {"scored_tokens": n_tokens}
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(out).encode())
    def log_message(self, *a):
        pass

def one_step(base_url, batch):
    rollouts = [[100 + j] * ROLLOUT_LEN for j in range(ROLLOUTS_PER_STEP)]
    scored = 0
    for i in range(0, ROLLOUTS_PER_STEP, batch):
        r = requests.post(f"{base_url}/v1/completions",
                          json={"model": "mock", "prompts": rollouts[i:i + batch],
                                "prompt_logprobs": 8}, timeout=60).json()
        scored += r["scored_tokens"]
    assert scored == ROLLOUTS_PER_STEP * ROLLOUT_LEN
    return scored

measured, predicted = {}, {}
with socketserver.TCPServer(("127.0.0.1", 0), MockScoringServer) as srv:
    url = f"http://127.0.0.1:{srv.server_address[1]}"
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    print(f"{'batch':>5} {'requests/step':>13} {'measured s/step':>16} "
          f"{'predicted s/step':>17} {'steps/sec':>10}")
    for b in (1, 4, 16):
        one_step(url, b)                                     # warmup
        t0 = time.perf_counter()
        for _ in range(3):
            one_step(url, b)
        measured[b] = (time.perf_counter() - t0) / 3
        n_req = ROLLOUTS_PER_STEP // b
        predicted[b] = n_req * (REQ_LATENCY_S + PER_TOKEN_S * b * ROLLOUT_LEN)
        print(f"{b:>5} {n_req:>13} {measured[b]:>16.3f} {predicted[b]:>17.3f} "
              f"{1/measured[b]:>10.2f}")
    srv.shutdown()

knee_b = REQ_LATENCY_S / (PER_TOKEN_S * ROLLOUT_LEN)
print(f"\nanalytic knee: per-request cost equals per-payload cost at batch "
      f"~{knee_b:.0f} rollouts/request")
assert measured[1] > measured[4] > measured[16], "amortization must be monotone"
for b in (1, 4, 16):
    assert measured[b] < 2.0 * predicted[b], f"batch {b}: overhead beyond the cost model"
    assert measured[b] > 0.9 * predicted[b], f"batch {b}: measured faster than possible"
assert measured[1] / measured[16] > 2.5, "batching 16x must buy several-fold step speed"
print("measured curve matches the known cost model: the harness, and the effect, are real")

batch requests/step  measured s/step  predicted s/step  steps/sec


    1            16            0.976             0.902       1.02


    4             4            0.329             0.302       3.04


   16             1            0.160             0.152       6.24



analytic knee: per-request cost equals per-payload cost at batch ~8 rollouts/request
measured curve matches the known cost model: the harness, and the effect, are real


In [7]:
# Gated: the same measurement against the real vLLM server, using the lab's
# A-2 score_remote client (one request per scoring batch) inside a training step.
if RUN_SERVER:
    SERVER = "http://127.0.0.1:8000"
    # For batch in (1, 4, 16): run 20 optimizer steps of the Lab 04 student where
    # teacher scores come from score_remote, sending `batch` rollouts per request
    # (concatenate rollouts into one prompt_logprobs call per group). Record
    # steps/sec and the fraction of step time inside requests.post, the number
    # the lab's B-3 says to log. Reconcile: the knee should sit where the
    # measured per-request latency (from this table) equals batch * rollout_len
    # divided by the B-2 prefill tok/s at that batch.
    pass
else:
    print("RUN_SERVER=False: the real-server measurement compiled but did not execute.")

RUN_SERVER=False: the real-server measurement compiled but did not execute.


**Interpretation.** The measured table shows the amortization curve the exercise asks for,
and the asserts hold it to the known cost model within tight bounds (under 2x above, never
below 90 percent of, the prediction; the excess over prediction is genuine HTTP and client
overhead, itself a small per-request cost that amortizes the same way). Reading the
numbers: at batch 1 a step pays the 50 ms request toll 16 times, about 0.9 seconds per
step; at batch 16 it pays once, about 0.15 seconds, a better than 5x step-rate improvement
with identical scored tokens. The knee has a closed form worth keeping: fixed-per-request
cost equals payload cost when batch times rollout length times per-token cost reaches the
request latency, here around 8 rollouts per request, which is why the gain from 1 to 4 is
large and the gain from 4 to 16 is smaller. Past the knee you are trading latency for
nothing; before it you are burning step time on tolls.

The reconciliation with B·2's prefill curve, which is what the exercise means by it: the
real server's per-token scoring cost is not a constant, it is one over the prefill tokens
per second *at that batch size*, and B·2 showed prefill throughput rising with batch until
compute saturates. So on the real server batching helps twice, once by amortizing the
request toll (this cell's effect) and once by densifying prefill (B·2's effect), and the
real knee sits where the request toll equals the batch's payload time computed from the
measured prefill curve, not from a constant. The gated cell (`RUN_SERVER=False`;
unexecuted) states that check. Expected live result, grounded in Part C: steps/sec rising
toward a plateau by batch 8 to 16, and the fraction of step time spent waiting on scoring
falling from dominant at batch 1 to a minority at batch 16; the Part C rule follows, batch
scoring across gradient-accumulation microbatches before considering any topology change.
Failure signature: steps/sec *falling* at batch 16 means the payload outgrew the server's
token budget per scheduling round (check `max_num_batched_tokens`), which is queueing, not
amortization, and shrinking to the knee fixes it.

## Exercise 4: The MoE audit

**The exercise, restated.** Back out the effective bytes read per decoded token from a
measured MoE throughput number and compare it to the spec-sheet estimate, active parameters
times bytes per parameter. The lab's published numbers: 49.7 tokens per second measured
decode for the 20B-class MXFP4 MoE model on 273 GB/s of bandwidth, a 10 GB weight
footprint on disk, and roughly 3.6B active parameters at half a byte each.

**The approach.** Fully live, because it is arithmetic on published measurements. The
roofline read backwards: if decode is bandwidth-bound, then bytes-per-token equals bandwidth
divided by throughput, no model of the internals required. That backed-out number is the
"dark bandwidth" figure, the bytes a token *actually* drags through memory as revealed by
the clock rather than by the spec sheet, and it must sit inside a chain of inequalities or
something in the story is wrong: above the active-slice estimate (the spec sheet counts only
expert weights, while the real token also reads the KV cache, attention working set, and
pays kernel inefficiency), and below the dense footprint (otherwise the routing is not
sparse at all and the MoE label is doing nothing). The same chain, stated as throughput,
is the lab's A·1 assertion pair, reproduced here as the closing link.

In [8]:
# Live: the audit, one division and a chain of inequalities.
measured_tps = 49.7          # tok/s, the published measured decode
dense_gb = 10.0              # GB on disk, the whole checkpoint
active_gb = 3.6 * 0.5        # GB per token IF only active experts were read: 1.8

effective_gb = BW / measured_tps            # bytes/token revealed by the clock
print(f"effective bytes per token : {BW:.0f} / {measured_tps} = {effective_gb:.2f} GB")
print(f"active-slice estimate     : 3.6B x 0.5 B/param      = {active_gb:.2f} GB")
print(f"dense footprint           :                           {dense_gb:.2f} GB")
print(f"overhead multiplier       : {effective_gb:.2f} / {active_gb:.2f} "
      f"= {effective_gb/active_gb:.2f}x the spec-sheet slice")
print(f"bandwidth efficiency      : {active_gb/effective_gb:.0%} of moved bytes are expert weights")

# The consistency chain. Each link is one claim about the architecture.
assert active_gb < effective_gb, \
    "the clock must reveal MORE than the active slice: KV, attention, and kernels are not free"
assert effective_gb < dense_gb, \
    "the clock must reveal LESS than the dense footprint: the routing really is sparse"
# The same chain expressed as throughput, the lab's A-1 pair, closed from this side:
dense_bound = bandwidth_bound_decode_tps(dense_gb, BW, bytes_per_param=1.0)
active_bound = bandwidth_bound_decode_tps(active_gb, BW, bytes_per_param=1.0)
assert dense_bound < measured_tps < active_bound, \
    "measured decode sits between the dense and active-slice rooflines"
# And the identity that makes the audit circular-proof: backing out throughput
# from the backed-out bytes returns the measurement exactly.
assert abs(BW / effective_gb - measured_tps) < 1e-9
print("\nconsistency chain holds: active slice < clock-revealed bytes < dense footprint")

effective bytes per token : 273 / 49.7 = 5.49 GB
active-slice estimate     : 3.6B x 0.5 B/param      = 1.80 GB
dense footprint           :                           10.00 GB
overhead multiplier       : 5.49 / 1.80 = 3.05x the spec-sheet slice
bandwidth efficiency      : 33% of moved bytes are expert weights

consistency chain holds: active slice < clock-revealed bytes < dense footprint


**Interpretation.** The audit closes the lab's A·1 loop with numbers. The clock says every
decoded token moves 5.49 GB (273 divided by 49.7), against a spec-sheet active slice of
1.80 GB, so the model is spending about 3x the theoretical minimum bytes per token, and
only about a third of the moved bytes are the expert weights the spec sheet counts. The
chain of asserts is the audit's logic made executable: 1.80 below 5.49 confirms the
spec-sheet number is an underestimate of real traffic, as it must be, since a real token
also reads the shared non-expert layers, the growing KV cache, and pays for imperfect
kernels and scheduling; 5.49 below 10.0 confirms the routing is genuinely sparse, because a
dense read of the checkpoint would cap throughput at 27.3 tokens per second and the
measured 49.7 would be impossible. That impossibility is exactly how A·1 detected the MoE
architecture from the outside, and this exercise runs the same reasoning in the opposite
direction, from measurement to bytes instead of from bytes to a bound.

What to do with the number: the 3.05x overhead multiplier is this platform's honest MoE
serving tax, and it, not the spec sheet, is what belongs in any Lab 06-style pricing table
for a MoE teacher (a SeqKD corpus priced at 1.8 GB per token would come in 3x optimistic).
What would count as refuting evidence when auditing a *different* MoE serve: an effective
bytes-per-token *below* the active slice, which no kernel can achieve and therefore means a
mis-measurement (usually timing launches instead of work, the harness rule from A·3) or a
wrong active-parameter count; or a value at or above the dense footprint, which means the
router is degenerate (all tokens to the same experts would give the opposite signature:
*below*-dense but with terrible load balance, visible only in server metrics, so the
byte audit and the server's own counters are complements, not substitutes).